# Importing Libraries

In [1]:
from umap import UMAP # for UMAP latent space projections
import sys # for relative imports of sigma

Using relative imports for sigma2 and its packages

In [2]:
sys.path.insert(0,"..")

from sigma.utils import normalisation as norm 
from sigma.utils import visualisation as visual
from sigma.utils.load import SEMDataset
from sigma.src.utils import same_seeds
from sigma.src.dim_reduction import Experiment
from sigma.models.autoencoder import AutoEncoder
from sigma.src.segmentation import PixelSegmenter
from sigma.gui import gui
from sigma.utils.loadtem import TEMDataset

# Loading the data

In [3]:
file_path='test.bcf'
sem=SEMDataset(file_path)

# Viewing the data

The dataset can be viewed with `gui.view_dataset`. In the gui, it is possilbe to view the summed spectra across the dataset, search for edges near a particular energy, and define edges of interest in the feature list.

Elemental maps for these features are shown in the 'Elemental maps' tab.

In [4]:
gui.view_dataset(sem)

Output()

Output()

# Processing steps before dimensionality reduction

To improve performance, it is often desireable to bin the signal in the navigation dimension.
This is done with `sem.rebin_signal(size=(nx,ny))`, which will bin the signal by nx,ny

The signals for each pixel are also normalised with `sem.peak_intensity_normalisation`

The initial 0 energy peak is removed from all signals with `sem.remove_first_peak(end=float)` where `end` specifies the energy at which the first peak ends

In [5]:
# Rebin both edx and bse dataset
sem.rebin_signal(size=(3,3))

# normalisation to make the spectrum of each pixel summing to 1.
sem.peak_intensity_normalisation()

# Remove the first peak until the energy of 0.1 keV

sem.remove_first_peak(end=0.1)


Rebinning the intensity with the size of (3, 3)
Normalising the chemical intensity along axis=2, so that the sum is equal to 1 along axis=2.
Removing the first peak by setting the intensity to zero until the energy of 0.1 keV.


In [6]:
#sem.normalisation([norm.neighbour_averaging])
sem.normalisation([norm.neighbour_averaging,norm.zscore])
#sem.normalisation([norm.neighbour_averaging,norm.zscore,norm.softmax])

Set feature_list to ['Al_Ka', 'C_Ka', 'Ca_Ka', 'Fe_Ka', 'K_Ka', 'O_Ka', 'Si_Ka', 'Ti_Ka', 'Zn_Ka']
Normalise dataset using:
    1. neighbour_averaging
    2. zscore


In [7]:
gui.view_pixel_distributions(sem,norm_list=[norm.neighbour_averaging,norm.zscore,norm.softmax],cmap='Reds')

In [8]:
print('After normalisation:')
gui.view_intensity_maps(spectra=sem.normalised_elemental_data, element_list=sem.feature_list)

After normalisation:


## Adding electron intensity as a component for projection

If you wish to add the BSE image (or other navigation image) to the feature vectors that are used for clustering, you can do so with 
`sem.get_feature_maps_with_nav_img()`

You will need to specify the normalisation steps you performed on the eds data on this image, as shown below

The bse image will be automatically resized to be the same dimensions as the eds maps


In [9]:
sem.get_feature_maps_with_nav_img(normalisation=[norm.neighbour_averaging,norm.zscore])

Resizing nav_img from (514, 279) to (171, 93)


In [10]:
gui.view_dataset(sem)

Output()

Output()

# Dimensionality Reduction

The above steps create an $x \times y \times n$ data cube. To perform clustering, it is necessary to project this into a lower dimensonal latent space

Dimensionality reduction may be performed either with the [UMAP algorithm](https://arxiv.org/abs/1802.03426) or with an autoencoder (more details on the autoencoder can be found [here](https://agupubs.onlinelibrary.wiley.com/doi/full/10.1029/2022GC010530) )

It is not necessary to run both dimensionality reductions, but both are demonstrated in this notebook for completeness.

## Latent Space Projection with UMAP

In [13]:
data = sem.normalised_elemental_data.reshape(-1,len(sem.feature_list))
umap = UMAP(
        n_neighbors=15,
        min_dist=0.02,
        n_components=2,
        metric='euclidean'
    )
latent = umap.fit_transform(data)

## Latent Space Projection with Autoencoder

In [12]:
# The integer in this function can determine different initialised parameters of model (tuning sudo randomness)
# This can influence the result of dimensionality reduction and change the latent space.
same_seeds(2)

# Set up the experiment, e.g. determining the model structure, dataset for training etc.
general_results_dir='./' 
ex = Experiment(descriptor='zscore',
                general_results_dir=general_results_dir,
                model=AutoEncoder,
                model_args={'hidden_layer_sizes':(512,256,128)}, # number of hidden layers and corresponding neurons
                chosen_dataset=sem.normalised_elemental_data,
                save_model_every_epoch=True)

model_name: Model-zscore
size_dataset: (171, 93, 10)
device: cpu
num_parameters: 343820


In [13]:
ex.run_model(num_epochs=50,
             patience=50, 
             batch_size=64,
             learning_rate=1e-4, 
             weight_decay=0.0, 
             task='train_all', # Change to 'train_eval' to train on the training set (85% dataset) and test on a testing set (15%) for evaluation
             noise_added=0.0,
             KLD_lambda=0.0,
             criterion='MSE',
             lr_scheduler_args={'factor':0.5,
                                'patience':5, 
                                'threshold':1e-2, 
                                'min_lr':1e-6,
                                'verbose':True}) 
latent = ex.get_latent()

num_epochs: 50
batch_size: 64
task: train_all
optimizer: lr=0.0001 and weight_decay=0.0

Start training ...



  0%|          | 0/249 [00:00<?, ?batch/s]

Epoch 1 ----> model saved, train_loss=0.801757 | test_loss = 0.801757


  0%|          | 0/249 [00:00<?, ?batch/s]

Epoch 2 ----> model saved, train_loss=0.759487 | test_loss = 0.759487


...

  0%|          | 0/249 [00:00<?, ?batch/s]

Epoch 00027: reducing learning rate of group 0 to 6.2500e-06.
Epoch 27 ----> model saved, train_loss=0.747105 | test_loss = 0.747105


  0%|          | 0/249 [00:00<?, ?batch/s]

  0%|          | 0/249 [00:00<?, ?batch/s]

KeyboardInterrupt: 

# Visualising the prjection

Before clustering, the results of the dimensionality reduction can quickly be visualised with `gui.show_projection`

In [11]:
gui.show_projection(latent)

NameError: name 'latent' is not defined

# Pixel Segmentation

Once a latent space is formed, it can be split into clusters using segmentation algorithms.

The first one demsonstated is Gaussian Mixture Modelling (GMM)

## Segmentation with GMM

First, create a `PixelSegmenter` object 

In [ ]:
ps_gmm=PixelSegmenter(latent=latent,
                      dataset=sem,method='GaussianMixture',
                      method_args={'n_components' :50,
                                   'random_state':0, 'init_params':'kmeans'} )

## Segmentation with HDBSCAN

Instead of GMM, clustering may be performed with [HDBSCAN](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.HDBSCAN.html)
HDBSCAN will leave several points as unassigned during the segmentation. The interactive latent plots can handle this, leaving them as grey coloured in the images, and they are automaticlly excluded when clusters are selected for merging.

It is possible to include the unassigned points when merging or creating new clusters by highlighting the "Include -1" button in the GUI.

First perform a HDBSCAN segmetation on the dataset by creating a new `PixelSegmenter` object

In [14]:
ps_hdb = PixelSegmenter(latent=latent, 
                    dataset=sem,
                    method="HDBSCAN",
                    method_args=dict(min_cluster_size=5, min_samples=10,
                                     max_cluster_size=int(len(latent)/10),
                                     cluster_selection_epsilon=1e-1) )

SIGMA includes many options for viewing the clustering. A simple tool for viewing the latent space is `view_latent_space`, which allows for initial inspection

In [16]:
gui.view_latent_space(ps=ps_hdb, color=True)

### Interacting with clusters in the latent space

It is possible to interact with latent space using the `interactive_latent_plot` command. 

Within this function, it is possible to 

1. Change the colour of a specific cluster.
2. Select multiple clusters, and make them all the same colour.
3. Merge multiple clusters together.
4. Select points to create a new cluster

First, demonstrate the recolouring of cluster(s).

To do this

1. Run the plot
2. Select clusters either with the rectangle, lasso, or by clicking them
3. Check the appropriate clusters are selected with the printout below the plot. If not, press 'Clear Selection'
4. Select a colour from the displayed colours at the top
5. Press the Recolour button

To Merge clusters

1. Ensure the selection is clear by pressing 'Clear Selection' GUI
2. Select the clusters to be merged using rectangle or lasso tools
3. Press the 'Merge Clusters' button

To create a new cluster

1. Press 'Select Points' to eneter point selection mode
2. For certain clustering algortihms such as HDBSCAN, some clusters are treated as noise and are labelled as -1. These can be included or exclued from the new cluster by pressing "Include -1" or "Exclude -1"
3. Press Create Cluster
4. The newly created cluster can be recoloured using the normal recolour functionality

In [16]:
gui.interactive_latent_plot(ps=ps_gmm,ratio_to_be_shown=1.,n_colours=30)

Once you are happy with the merging, creating and recolouring, the updated PixelSegmenter object can be saved

In [17]:
ps_gmm.save_state('gmm_merged_with_bse.pkl')

✅ Saved state to gmm_merged_with_bse.pkl


Another tool to inspect the latent space is `check_latent_space`, which contains information about where the clusters are located on the navigation image.

In [18]:
gui.check_latent_space(ps=ps_gmm,show_map=True,ratio_to_be_shown=1.0,alpha_cluster_map=0.5)

alt.HConcatChart(...)

# Loading a saved `PixelSegmenter` Object
The saved object contains information about the latent space and clusters, but does not contain the base dataset. Therefore, the basedataset must be re-loaded (and re-processed) before the `PixelSegmenter` object can be loaded

Reloading the sem dataset

In [19]:
file_path='test.bcf'

sem_loaded=SEMDataset(file_path) #loading it in again

sem_loaded.rebin_signal(size=(4,4)) #repeating the processing steps from earlier
sem_loaded.peak_intensity_normalisation()
sem_loaded.remove_first_peak(end=0.1)

sem_loaded.normalisation([norm.neighbour_averaging,norm.zscore])

#sem_loaded.get_feature_maps_with_nav_img(normalisation=[norm.neighbour_averaging,norm.zscore])


Rebinning the intensity with the size of (4, 4)
Normalising the chemical intensity along axis=2, so that the sum is equal to 1 along axis=2.
Removing the first peak by setting the intensity to zero until the energy of 0.1 keV.
Set feature_list to ['Al_Ka', 'C_Ka', 'Ca_Ka', 'Fe_Ka', 'K_Ka', 'O_Ka', 'Si_Ka', 'Ti_Ka', 'Zn_Ka']
Normalise dataset using:
    1. neighbour_averaging
    2. zscore


Now the ps object can be loaded

In [20]:
ps_loaded = PixelSegmenter.from_saved_state("gmm_merged_with_bse.pkl",dataset=sem_loaded)

Check the loaded object is what we saved

In [21]:
gui.interactive_latent_plot(ps=ps_loaded,ratio_to_be_shown=1.0,n_colours=30)


`view_latent_space` , `interactive_latent_plot` and `check_latent_space` can all be used to inspect and interact with the clustering from HDBSCAN.

In [22]:
gui.interactive_latent_plot(ps=ps_hdb,ratio_to_be_shown=1.0,)

The customised clusters object from HDBSCAN can be saved and loaded the same way as the GMM clusters object.

# Viewing the results of the clustering

For improved visualtion of the contents of each cluster, the following functions are available

`view_phase_map` shows the clusters overlayed on the navigation image

In [17]:
gui.view_phase_map(ps=ps_gmm,alpha_cluster_map=0.5)

`show_cluster_distribution` shows the distribution of each (or a specific) cluster, along with its "feature vector" (which describes the elements that cluster is rich or deficient in) and the summed spectra for that cluster

In [18]:
gui.show_cluster_distribution(ps=ps_gmm)

Output()

`view_clusters_sum_spectra` shows a similar plot, but with clusters overlayed on the navigation image. This function also allows for the summed spectra for each cluster to be plotted in a more interactive manner by selecting a specific cluster from the dropdown menu then opening the spectra tab

In [19]:
gui.view_clusters_sum_spectra(ps=ps_gmm, normalisation=True, spectra_range=(0,8))

SelectMultiple(options=('cluster_0', 'cluster_1', 'cluster_2', 'cluster_3', 'cluster_4', 'cluster_5', 'cluster…

Output()

## Exporting the cluster masks

Binary cluster masks may be exported as `.tif` files for each of the clusters in the ps object.

This is done with the `export_cluster_masks` method.

These masks may be upscaled to a high resolution image if desired by passing a path to the `high_res_path` argument

In [20]:
ps_gmm.export_cluster_masks(output_dir='cluster_masks') # export at default resolution of the navigation image

Exported 50 cluster mask(s) to 'cluster_masks'.


In [16]:
ps_hdb.export_cluster_masks(output_dir='high_res_cluster_masks',high_res_path='high_res_image.tif') # export at resolution of specified image

FileNotFoundError: No such file: 'C:\Users\Tom\Documents\PostDOc\Sigma2\SIGMA2\tutorials\high_res_image.tif'

## Cluster Statistics

There are methods of investigating the statistics of the clustering with SIGMA

* ``show_cluster_statistics`` - performs a "particle analysis" type result, with Areas, ECDs and other stats about each cluster
*  ``show_cluster_proportions`` - gives basic information about what proportion of the image is in each cluster

In [ ]:
gui.show_cluster_stats(ps_gmm)

In [ ]:
gui.show_cluster_proportions(ps_gmm)


# Perform NMF

Once clustering is complete, the individual spectra that contribute to the overall spectra from each cluster may be estimated using Non-negative Matrix Factorisation (NMF)

More details of the methodology can be found [here](https://agupubs.onlinelibrary.wiley.com/doi/full/10.1029/2022GC010530)

First, perform NMF to get the weights of the constituent spectra, and the spectra themselves (components)

This is demonstrated assuming there are three components

In [17]:
weights, components = ps_hdb.get_unmixed_spectra_profile(clusters_to_be_calculated='All', 
                                                 n_components=4,
                                                 normalised=False, 
                                                 method='NMF', 
                                                 method_args={'init':'nndsvd'})

In [18]:
weights_guessed, components_guessed = ps_gmm.get_unmixed_spectra_profile_init_guess(clusters_to_be_calculated='All', 
                                                 n_components=3,
                                                 normalised=False, 
                                                 method='NMF', 
                                                 method_args={'init':'nndsvd'},
                                                seed_clusters=[1,4,3])

NameError: name 'ps_gmm' is not defined

## Visualising the results of NMF

The results can be inspected with `show_unmixed_weights_and_components`. 

The contibutions of the components to each cluster can be shown by selecting a specific cluster and visualising in the single weight tab

The (3 in this case) component spectra can be seen in the All components tab

A single component can be selected, and inspected in more detail in the Single component tab

In [ ]:
gui.show_unmixed_weights_and_compoments(ps=ps_hdb, weights=weights, components=components)

An RGB map of the component spectra can be plotted with `show_abundance_map`

In [ ]:
gui.show_abundance_map(ps=ps_hdb, weights=weights, components=components)

# quantitative NMF 
still in development. use kfactors to adjust the CL factors used in quantification

In [20]:
weights_df, components_df, integrated_intensities, compositions_df, mol_fractions_df, phase_sensitivity = \
    ps_hdb.get_unmixed_spectra_profile_quant(
        clusters_to_be_calculated="All",
        n_components=4,
        normalised=False,
        method="NMF",
        method_args={"init": "nndsvd"},
        elements=["O", "Mg", "Al", "Si", "Ca", "Fe", "Ni"],
        background_method="linear",
    )

print("=== Signal mixing fractions ===")
display(weights_df)

print("=== Endmember compositions (wt. fraction) ===")
display(compositions_df)

print("=== Mol fractions per cluster ===")
display(mol_fractions_df)

print("=== Phase sensitivity factors ===")
display(phase_sensitivity.to_frame("S_p"))

print(f"\nNMF reconstruction error: {ps_hdb.NMF_recon_error:.6f}")

=== Signal mixing fractions ===


,cpnt_0,cpnt_1,cpnt_2,cpnt_3
cluster_0,0.0324,0.0541,0.5332,0.3803
cluster_1,0.2069,0.0908,0.3731,0.3292
cluster_2,0.1656,0.0627,0.5212,0.2505
cluster_3,0.0017,0.6242,0.0166,0.3575
cluster_4,0.6430,0.1219,0.0535,0.1816
...,...,...,...,...
cluster_193,0.7978,0.0168,0.0089,0.1765
cluster_194,0.8017,0.0195,0.0131,0.1656
cluster_195,0.8223,0.0171,0.0036,0.1570
cluster_196,0.7817,0.0150,0.0064,0.1969


=== Endmember compositions (wt. fraction) ===


,O,Mg,Al,Si,Ca,Fe,Ni
cpnt_0,0.0247,0.0015,0.0057,0.0004,0.9543,0.0101,0.0033
cpnt_1,0.0484,0.0013,0.2009,0.7151,0.0176,0.0140,0.0028
cpnt_2,0.0327,0.0012,0.0135,0.0061,0.0313,0.9124,0.0028
cpnt_3,0.4240,0.0141,0.1063,0.1700,0.2615,0.0151,0.0090


=== Mol fractions per cluster ===


,cpnt_0,cpnt_1,cpnt_2,cpnt_3
cluster_0,0.0055,0.0212,0.3284,0.6449
cluster_1,0.0411,0.0414,0.2675,0.6499
cluster_2,0.0354,0.0308,0.4019,0.5319
cluster_3,0.0003,0.2839,0.0119,0.7038
cluster_4,0.2203,0.0958,0.0661,0.6178
...,...,...,...,...
cluster_193,0.3043,0.0147,0.0122,0.6687
cluster_194,0.3158,0.0176,0.0186,0.6479
cluster_195,0.3379,0.0161,0.0053,0.6407
cluster_196,0.2797,0.0123,0.0083,0.6997


=== Phase sensitivity factors ===


,S_p
cpnt_0,1923.876712
cpnt_1,838.357295
cpnt_2,533.401664
cpnt_3,193.706141



NMF reconstruction error: 28.554199
